# Laboratorio 1: punto flotante, error y estabilidad

**Curso:** Métodos Numéricos -- Universidad del Pacífico  
**Periodo:** 2026-2  
**Duración sugerida:** 115 minutos

## Propósito

Al finalizar el laboratorio podrá:

1. inspeccionar la precisión y el rango de `float64`;
2. distinguir error de truncamiento y error de redondeo;
3. reproducir no asociatividad, overflow, underflow y cancelación;
4. medir errores absoluto, relativo, hacia adelante y hacia atrás;
5. aplicar análisis diferencial para estimar la propagación de errores;
6. distinguir condicionamiento del problema y estabilidad del algoritmo;
7. comparar exactitud y costo de implementaciones alternativas.

## Material usado como base

- `SesionSemana1Sesión1.ipynb`: introducción a Python y NumPy.
- `Lab_QR_Householder_Givens_Cholesky_v3.ipynb`: estructura de objetivos,
  experimentos reproducibles y preguntas.

## Referencias de NumPy

- [`numpy.finfo`](https://numpy.org/doc/stable/reference/generated/numpy.finfo.html)
- [`numpy.expm1`](https://numpy.org/doc/stable/reference/generated/numpy.expm1.html)
- [`numpy.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html)
- [`numpy.sum`](https://numpy.org/doc/stable/reference/generated/numpy.sum.html)

## 0. Preparación

El laboratorio utiliza únicamente Python, NumPy y módulos de la biblioteca
estándar. Ejecute las celdas en orden. Los resultados temporales de medición de
tiempo dependen del equipo y deben interpretarse comparativamente.

In [1]:
import itertools
import math
import platform
import timeit
from decimal import Decimal, getcontext

import numpy as np

np.set_printoptions(precision=17, suppress=False)
getcontext().prec = 60

print("Python:", platform.python_version())
print("NumPy:", np.__version__)

Python: 3.12.13
NumPy: 2.3.5


## 1. Anatomía de `float64`

`np.finfo` informa los límites de un tipo de punto flotante. Distinguiremos:

- `eps`: distancia entre 1 y el siguiente número representable mayor que 1;
- `smallest_normal`: menor número positivo normalizado;
- `smallest_subnormal`: menor número positivo representable;
- `max`: mayor número finito representable.

In [2]:
info = np.finfo(np.float64)
print(f"bits                 = {info.bits}")
print(f"eps                  = {info.eps:.18e}")
print(f"u = eps/2            = {info.eps/2:.18e}")
print(f"smallest_normal      = {info.smallest_normal:.18e}")
print(f"smallest_subnormal   = {info.smallest_subnormal:.18e}")
print(f"max                  = {info.max:.18e}")

bits                 = 64
eps                  = 2.220446049250313081e-16
u = eps/2            = 1.110223024625156540e-16
smallest_normal      = 2.225073858507201383e-308
smallest_subnormal   = 4.940656458412465442e-324
max                  = 1.797693134862315708e+308


In [3]:
valores = [1.0, 1e8, 1e16]
print("Separación local entre números representables")
for x in valores:
    siguiente = np.nextafter(x, np.inf)
    print(f"x={x:>8.1e}  spacing={np.spacing(x):.18e}  siguiente-x={siguiente-x:.18e}")

Separación local entre números representables
x= 1.0e+00  spacing=2.220446049250313081e-16  siguiente-x=2.220446049250313081e-16
x= 1.0e+08  spacing=1.490116119384765625e-08  siguiente-x=1.490116119384765625e-08
x= 1.0e+16  spacing=2.000000000000000000e+00  siguiente-x=2.000000000000000000e+00


### Preguntas 1

1. ¿Por qué `spacing(1e16)` es mucho mayor que `spacing(1.0)`?
2. ¿Cuál es la diferencia conceptual entre `smallest_normal` y
   `smallest_subnormal`?
3. Verifique experimentalmente que `1.0 + eps > 1.0`, pero
   `1.0 + eps/2 == 1.0` bajo el redondeo usual.

In [4]:
print("1 + eps   > 1:", 1.0 + info.eps > 1.0)
print("1 + eps/2 = 1:", 1.0 + info.eps/2 == 1.0)

1 + eps   > 1: True
1 + eps/2 = 1: True


## 2. Representación decimal y no asociatividad

El decimal 0.1 no tiene una expansión binaria finita. Python almacena el
`float64` más cercano. `float.hex()` permite observar su representación binaria
normalizada en formato hexadecimal.

In [5]:
x = 0.1
print("repr(0.1)              =", repr(x))
print("0.1 en hexadecimal     =", x.hex())
print("0.1 + 0.2              =", format(0.1 + 0.2, ".17g"))
print("¿Es 0.1+0.2 igual 0.3? =", 0.1 + 0.2 == 0.3)

decimal_exacta = Decimal("0.1")
decimal_del_float = Decimal.from_float(x)
print("Error de representación =", decimal_del_float - decimal_exacta)

repr(0.1)              = 0.1
0.1 en hexadecimal     = 0x1.999999999999ap-4
0.1 + 0.2              = 0.30000000000000004
¿Es 0.1+0.2 igual 0.3? = False
Error de representación = 5.5511151231257827021181583404541015625E-18


In [6]:
a, b, c = 1e16, -1e16, 1.0
izquierda = (a + b) + c
derecha = a + (b + c)
print("(a+b)+c =", izquierda)
print("a+(b+c) =", derecha)
print("Diferencia =", izquierda - derecha)

(a+b)+c = 1.0
a+(b+c) = 0.0
Diferencia = 1.0


### Preguntas 2

1. Explique en qué operación se pierde el valor 1.
2. ¿La identidad asociativa es falsa en números reales o falla su realización
   mediante operaciones de máquina?
3. Pruebe las seis permutaciones de `(1e16, 1.0, -1e16)` y registre los
   resultados distintos.

In [7]:
datos = (1e16, 1.0, -1e16)
for p in itertools.permutations(datos):
    print(p, "->", sum(p))

(1e+16, 1.0, -1e+16) -> 1.0
(1e+16, -1e+16, 1.0) -> 1.0
(1.0, 1e+16, -1e+16) -> 1.0
(1.0, -1e+16, 1e+16) -> 1.0
(-1e+16, 1e+16, 1.0) -> 1.0
(-1e+16, 1.0, 1e+16) -> 1.0


## 3. Truncamiento y redondeo al aproximar una derivada

Para $f(x)=e^x$, aproximaremos $f'(0)=1$ mediante

$$
D_hf(0)=\frac{f(h)-f(0)}{h}=\frac{e^h-1}{h}.
$$

Con $h$ grande domina el truncamiento. Con $h$ extremadamente pequeño, la
resta `exp(h)-1` pierde cifras y domina el redondeo.

In [8]:
def derivada_adelante(h):
    return (np.exp(h) - 1.0) / h

hs = np.logspace(-1, -16, 16)
errores_derivada = np.abs(np.array([derivada_adelante(h) for h in hs]) - 1.0)
mejor = int(np.argmin(errores_derivada))

print(f"{'h':>12} {'aproximación':>18} {'error absoluto':>18}")
for i in [0, 3, 6, 9, 12, 15]:
    aprox = derivada_adelante(hs[i])
    print(f"{hs[i]:12.1e} {aprox:18.12e} {errores_derivada[i]:18.3e}")
print(f"\nMejor h de la malla: {hs[mejor]:.1e}")

           h       aproximación     error absoluto
     1.0e-01 1.051709180756e+00          5.171e-02
     1.0e-04 1.000050001667e+00          5.000e-05
     1.0e-07 1.000000049434e+00          4.943e-08
     1.0e-10 1.000000082740e+00          8.274e-08
     1.0e-13 9.992007221626e-01          7.993e-04
     1.0e-16 0.000000000000e+00          1.000e+00

Mejor h de la malla: 1.0e-08


### Preguntas 3

1. ¿Por qué el error disminuye al principio y después vuelve a crecer?
2. Sustituya `np.exp(h)-1` por `np.expm1(h)`. ¿Qué parte del error cambia?
3. Explique por qué tomar el menor $h$ disponible no es una regla válida.

## 4. Análisis diferencial del error

Para $y=f(x_1,\ldots,x_n)$ y perturbaciones pequeñas, la aproximación de
primer orden es

$$
\Delta y\approx\sum_j\frac{\partial f}{\partial x_j}\Delta x_j.
$$

En términos relativos, cada factor $(x_j/y)(\partial f/\partial x_j)$
mide la amplificación local del error de la entrada $x_j$. Verificaremos esta
aproximación con $Y=A K^{\alpha}L^{\beta}$.

In [9]:
A, K, L = 1.0, 100.0, 80.0
alpha, beta = 0.35, 0.65
eps_A, eps_K, eps_L = 0.0, 0.01, 0.02

def produccion(A, K, L):
    return A * K**alpha * L**beta

Y = produccion(A, K, L)
Y_perturbado = produccion(A*(1+eps_A), K*(1+eps_K), L*(1+eps_L))
error_relativo_observado = (Y_perturbado - Y) / Y
error_relativo_diferencial = eps_A + alpha*eps_K + beta*eps_L

print(f"Error relativo diferencial: {error_relativo_diferencial:.8%}")
print(f"Error relativo observado:    {error_relativo_observado:.8%}")
print(f"Diferencia de orden superior: {abs(error_relativo_observado-error_relativo_diferencial):.3e}")

Error relativo diferencial: 1.65000000%
Error relativo observado:    1.64887875%
Diferencia de orden superior: 1.121e-05


### Preguntas 4

1. ¿Por qué la predicción diferencial no coincide exactamente con el cambio observado?
2. Reduzca las perturbaciones por un factor de diez. ¿Cómo cambia la diferencia?
3. Cambie el signo de una perturbación. ¿Cuándo se compensan parcialmente los errores?
4. ¿Qué entrada domina la incertidumbre cuando $\beta>\alpha$?

## 5. Error absoluto, relativo, hacia adelante y hacia atrás

Para un valor exacto $x$ y una aproximación $\widehat{x}$:

$$
E_{\mathrm{abs}}=|x-\widehat{x}|,
\qquad
E_{\mathrm{rel}}=\frac{|x-\widehat{x}|}{|x|}.
$$

En una ecuación $F(x)=0$, el residuo es $|F(\widehat{x})|$. El residuo no es el
mismo objeto que el error hacia adelante, aunque ambos pueden relacionarse con
el condicionamiento.

In [10]:
def errores(exacto, aproximado):
    absoluto = abs(exacto - aproximado)
    relativo = absoluto / abs(exacto) if exacto != 0 else np.nan
    return absoluto, relativo

exacto = math.sqrt(2.0)
aproximado = 1.4142
e_abs, e_rel = errores(exacto, aproximado)
residuo = abs(aproximado**2 - 2.0)

print(f"error absoluto = {e_abs:.8e}")
print(f"error relativo = {e_rel:.8e}")
print(f"residuo        = {residuo:.8e}")

error absoluto = 1.35623731e-05
error relativo = 9.59004598e-06
residuo        = 3.83600000e-05


In [11]:
# Error hacia atrás para el problema x^2 = d:
# ¿qué dato d_hat hace exacta la respuesta aproximada?
d = 2.0
d_hat = aproximado**2
backward_abs = abs(d_hat - d)
backward_rel = backward_abs / abs(d)

print(f"dato original d       = {d:.12f}")
print(f"dato perturbado d_hat = {d_hat:.12f}")
print(f"error relativo atrás  = {backward_rel:.8e}")

dato original d       = 2.000000000000
dato perturbado d_hat = 1.999961640000
error relativo atrás  = 1.91800000e-05


## 6. Cancelación y reformulación estable

Para $x$ grande, la expresión

$$
\sqrt{x+1}-\sqrt{x}
$$

resta cantidades casi iguales. La forma racionalizada

$$
\frac{1}{\sqrt{x+1}+\sqrt{x}}
$$

es algebraicamente equivalente, pero numéricamente más estable.

In [12]:
def diferencia_directa(x):
    return np.sqrt(x + 1.0) - np.sqrt(x)

def diferencia_estable(x):
    return 1.0 / (np.sqrt(x + 1.0) + np.sqrt(x))

print(f"{'x':>12} {'directa':>22} {'estable':>22} {'error relativo':>18}")
for x in [1.0, 1e4, 1e8, 1e12, 1e16]:
    directa = diferencia_directa(x)
    estable = diferencia_estable(x)
    rel = abs(directa-estable)/abs(estable)
    print(f"{x:12.1e} {directa:22.15e} {estable:22.15e} {rel:18.8e}")

           x                directa                estable     error relativo
     1.0e+00  4.142135623730951e-01  4.142135623730951e-01     1.34015774e-16
     1.0e+04  4.999875006248544e-03  4.999875006249609e-03     2.13029368e-13
     1.0e+08  5.000000055588316e-05  4.999999987500000e-05     1.36176633e-08
     1.0e+12  5.000038072466850e-07  4.999999999998749e-07     7.61449362e-06
     1.0e+16  0.000000000000000e+00  5.000000000000000e-09     1.00000000e+00


In [13]:
print(f"{'x':>12} {'exp(x)-1':>22} {'expm1(x)':>22} {'log(1+x)':>22} {'log1p(x)':>22}")
for x in [1e-4, 1e-8, 1e-12, 1e-16]:
    print(f"{x:12.1e} {np.exp(x)-1:22.15e} {np.expm1(x):22.15e} "
          f"{np.log(1+x):22.15e} {np.log1p(x):22.15e}")

           x               exp(x)-1               expm1(x)               log(1+x)               log1p(x)
     1.0e-04  1.000050001667141e-04  1.000050001666708e-04  9.999500033329732e-05  9.999500033330834e-05
     1.0e-08  9.999999939225290e-09  1.000000005000000e-08  9.999999889225291e-09  9.999999950000001e-09
     1.0e-12  1.000088900582341e-12  1.000000000000500e-12  1.000088900581841e-12  9.999999999995000e-13
     1.0e-16  0.000000000000000e+00  1.000000000000000e-16  0.000000000000000e+00  1.000000000000000e-16


### Preguntas 5

1. ¿A partir de qué escala la fórmula directa de los radicales pierde todas las
   cifras del resultado?
2. Explique por qué `expm1` y `log1p` son preferibles para argumentos pequeños.
3. Proponga otra expresión susceptible de cancelación y una reformulación.

## 7. El orden de suma es una decisión algorítmica

Compararemos suma secuencial, `np.sum` y `math.fsum`. La documentación de NumPy
indica que `np.sum` suele emplear suma parcial por pares; el detalle depende del
eje y la disposición de memoria. `math.fsum` utiliza una estrategia más lenta
orientada a mayor precisión.

In [14]:
def suma_secuencial(x):
    s = 0.0
    for valor in x:
        s += float(valor)
    return s

casos = {
    "orden original": [1e16, 1.0, -1e16],
    "cancelar primero": [1e16, -1e16, 1.0],
    "pequeños primero": [1.0, 1e16, -1e16],
}

for nombre, x in casos.items():
    print(f"{nombre:18s} secuencial={suma_secuencial(x):.1f}  "
          f"np.sum={np.sum(x):.1f}  math.fsum={math.fsum(x):.1f}")

orden original     secuencial=0.0  np.sum=0.0  math.fsum=1.0
cancelar primero   secuencial=1.0  np.sum=1.0  math.fsum=1.0
pequeños primero   secuencial=0.0  np.sum=0.0  math.fsum=1.0


## 8. Condicionamiento escalar

Para $y=f(x)$, el número de condición relativo local es

$$
\kappa_f(x)=\left|\frac{x f'(x)}{f(x)}\right|.
$$

Usaremos $f(x)=1/(1-x)$, cuyo condicionamiento crece cuando $x$ se aproxima a 1.

In [15]:
def f_sensible(x):
    return 1.0 / (1.0 - x)

def kappa_teorica(x):
    return abs(x / (1.0 - x))

def amplificacion_observada(f, x, perturbacion_rel=1e-8):
    dx = perturbacion_rel * x
    entrada_rel = abs(dx / x)
    salida_rel = abs((f(x + dx) - f(x)) / f(x))
    return salida_rel / entrada_rel

print(f"{'x':>8} {'kappa teórica':>18} {'amplificación observada':>26}")
for x in [0.1, 0.9, 0.99, 0.999]:
    print(f"{x:8.3f} {kappa_teorica(x):18.6f} {amplificacion_observada(f_sensible,x):26.6f}")

       x      kappa teórica    amplificación observada
   0.100           0.111111                   0.111111
   0.900           9.000000                   9.000001
   0.990          99.000000                  99.000098
   0.999         999.000000                 999.009985


## 9. Estabilidad: dos algoritmos para $a^2-b^2$

Compararemos

$$
\text{A: } a^2-b^2,
\qquad
\text{B: }(a-b)(a+b).
$$

Usaremos enteros de Python como referencia exacta y convertiremos los datos a
`float64` para evaluar ambos algoritmos. Antes de ejecutar la celda:

1. escriba los conjuntos de operandos $x^{(i)}$ de cada ruta;
2. identifique los mapas elementales cuya composición produce $a^2-b^2$;
3. separe en la aproximación diferencial el error de entrada y los redondeos locales;
4. explique por qué las dos rutas comparten el primer aporte, pero no el segundo.

In [16]:
a_entero = 100_000_001
b_entero = 100_000_000
referencia = a_entero*a_entero - b_entero*b_entero

a = np.float64(a_entero)
b = np.float64(b_entero)
algoritmo_A = a*a - b*b
algoritmo_B = (a-b)*(a+b)

print("referencia exacta =", referencia)
print("algoritmo A       =", algoritmo_A, "error =", algoritmo_A-referencia)
print("algoritmo B       =", algoritmo_B, "error =", algoritmo_B-referencia)

referencia exacta = 200000001
algoritmo A       = 200000000.0 error = -1.0
algoritmo B       = 200000001.0 error = 0.0


## 10. Costo de implementación: bucle y vectorización

La siguiente medición no es un benchmark universal. Sirve para comparar dos
capas de ejecución en el mismo equipo. Repita el experimento y reporte la
mediana de varias mediciones.

In [17]:
rng = np.random.default_rng(2026)
n = 100_000
u = rng.standard_normal(n)
v = rng.standard_normal(n)

def producto_bucle(u, v):
    s = 0.0
    for ui, vi in zip(u, v):
        s += ui * vi
    return s

referencia_bucle = producto_bucle(u, v)
referencia_numpy = np.dot(u, v)

t_bucle = min(timeit.repeat(lambda: producto_bucle(u, v), number=1, repeat=3))
t_numpy = min(timeit.repeat(lambda: np.dot(u, v), number=20, repeat=3)) / 20

print(f"producto con bucle = {referencia_bucle:.12e}")
print(f"producto con NumPy = {referencia_numpy:.12e}")
print(f"diferencia absoluta = {abs(referencia_bucle-referencia_numpy):.3e}")
print(f"tiempo bucle         = {t_bucle:.6f} s")
print(f"tiempo NumPy         = {t_numpy:.6f} s")
print(f"razón de tiempos     = {t_bucle/t_numpy:.1f}")

producto con bucle = -6.447493900075e+01
producto con NumPy = -6.447493900075e+01
diferencia absoluta = 5.969e-13
tiempo bucle         = 0.008844 s
tiempo NumPy         = 0.000003 s
razón de tiempos     = 2577.4


## 11. Reto integrador: auditoría numérica

Elija **uno** de los siguientes problemas:

1. evaluar $\sqrt{x+1}-\sqrt{x}$ para $x\in\{10^4,10^8,10^{12},10^{16}\}$;
2. calcular $e^x-1$ para $x\in\{10^{-4},10^{-8},10^{-12},10^{-16}\}$;
3. calcular $\log(1+x)$ para $x\in\{10^{-4},10^{-8},10^{-12},10^{-16}\}$;
4. aproximar $f'(0)$ para $f(x)=e^x$ variando $h$ entre $10^{-1}$ y $10^{-16}$;
5. contrastar la predicción diferencial de Cobb-Douglas con perturbaciones de distinta magnitud.

Entregue una tabla que contenga:

- algoritmo directo y algoritmo alternativo;
- valor de referencia;
- error absoluto y relativo;
- residuo o error hacia atrás, cuando corresponda;
- número de condición o una medida de sensibilidad;
- tiempo de ejecución para un tamaño de prueba justificado;
- recomendación final en no más de cinco líneas.

In [18]:
def reporte_error(nombre, aproximado, referencia):
    absoluto = abs(aproximado - referencia)
    relativo = absoluto / abs(referencia) if referencia != 0 else np.nan
    print(f"{nombre:20s} valor={aproximado:.17e}  "
          f"E_abs={absoluto:.3e}  E_rel={relativo:.3e}")

# Ejemplo de uso para el primer reto
x = 1e12
ref = diferencia_estable(x)
reporte_error("fórmula directa", diferencia_directa(x), ref)
reporte_error("fórmula estable", diferencia_estable(x), ref)

fórmula directa      valor=5.00003807246685028e-07  E_abs=3.807e-12  E_rel=7.614e-06
fórmula estable      valor=4.99999999999874934e-07  E_abs=0.000e+00  E_rel=0.000e+00


## 12. Cierre

Antes de aceptar un resultado numérico, pregunte:

1. ¿qué tipo y escala tienen los datos?;
2. ¿qué precisión y rango usa la máquina?;
3. ¿hay error de truncamiento y cómo cambia con el parámetro de aproximación?;
4. ¿qué propagación predice el análisis diferencial y cuándo es válida?;
5. ¿el algoritmo introduce cancelación o acumulación innecesaria?;
6. ¿qué muestran el residuo, el error hacia atrás y una referencia independiente?;
7. ¿el costo es razonable para la precisión obtenida?

**Producto esperado:** cuaderno ejecutado, respuestas breves a las preguntas y
el reto integrador documentado.